# RentGuard AI: Autoencoder Anomaly Detection & Underwriting Intelligence

This notebook demonstrates the end-to-end Machine Learning pipeline for **RentGuard AI**:
1. **Synthetic Rental Dataset Ingestion & EDA**
2. **Data Preprocessing Pipeline** (`ColumnTransformer` with `StandardScaler` and `OneHotEncoder`)
3. **Deep Learning Autoencoder Architecture** (Encoder $\to$ Latent Space $\to$ Decoder)
4. **Unsupervised Training** on representative normal applications
5. **Anomaly Threshold Calibration** (P95 and P99 percentiles on validation split)
6. **Evaluation on Held-Out Test Benchmark** (ROC-AUC, PR-AUC, Confusion Matrix)
7. **Feature-wise Error Attribution** for explainable underwriting

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# Ensure project root in python path
sys.path.append(os.path.abspath('..'))

from src.data_generator import create_rental_dataset
from src.preprocessing import RentalPreprocessor, ALL_INPUT_FEATURES
from src.train_model import build_autoencoder
from src.anomaly_detection import AnomalyDetector

## 1. Load Synthetic Rental Application Dataset
We load or generate 8,000 synthetic applications containing realistic financial correlations and ~8% controlled outliers.

In [ ]:
dataset_path = '../dataset/rental_applications.csv'
if not os.path.exists(dataset_path):
    df = create_rental_dataset(total_samples=8000, anomaly_rate=0.08, output_path=dataset_path)
else:
    df = pd.read_csv(dataset_path)

print(f"Total dataset shape: {df.shape}")
print(f"Normal count: {(df['is_anomaly'] == 0).sum()} | Anomaly count: {(df['is_anomaly'] == 1).sum()}")
df.head()

## 2. Partitioning: Pure Normal Training & Ground-Truth Test Benchmark
To adhere strictly to unsupervised anomaly detection principles, the model is trained exclusively on normal tenancy records.

In [ ]:
df_normal = df[df['is_anomaly'] == 0].copy()
df_anomaly = df[df['is_anomaly'] == 1].copy()

# 80% train+val, 20% test for normal records
normal_train_val, normal_test = train_test_split(df_normal, test_size=0.20, random_state=42)
normal_train, normal_val = train_test_split(normal_train_val, test_size=0.20, random_state=42)

# Combined test set with ground truth labels for benchmarking
df_test = pd.concat([normal_test, df_anomaly], ignore_index=True)
print(f"Train size: {len(normal_train)} | Val size: {len(normal_val)} | Test Benchmark size: {len(df_test)}")

## 3. Fit Preprocessing Pipeline
Fit `RentalPreprocessor` strictly on `normal_train` to prevent data leakage.

In [ ]:
preprocessor = RentalPreprocessor()
preprocessor.fit(normal_train)

X_train = preprocessor.transform(normal_train)
X_val = preprocessor.transform(normal_val)
X_test = preprocessor.transform(df_test)

print(f"Transformed input dimension: {X_train.shape[1]}")
print(f"Transformed feature names: {preprocessor.feature_names_out}")

## 4. Build & Train Deep Autoencoder
Architecture:
$$\text{Input (13)} \to \text{Dense(32)} \to \text{Dense(16)} \to \text{Latent(8)} \to \text{Dense(16)} \to \text{Dense(32)} \to \text{Output (13)}$$

In [ ]:
import keras

autoencoder = build_autoencoder(input_dim=X_train.shape[1], latent_dim=8)
autoencoder.summary()

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)
]

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=35,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

## 5. Threshold Calibration on Validation Normal Distribution
We calculate reconstruction error (Mean Squared Error) on the validation set and establish operational cutoffs at the 95th and 99th percentiles.

In [ ]:
val_reconstructions = autoencoder.predict(X_val)
val_mse = np.mean(np.square(X_val - val_reconstructions), axis=1)

p95 = np.percentile(val_mse, 95)
p99 = np.percentile(val_mse, 99)

print(f"Validation Normal Reconstruction Error Stats:")
print(f"Median (P50): {np.percentile(val_mse, 50):.4f}")
print(f"Medium Threshold (P95 Cutoff): {p95:.4f}")
print(f"High Threshold (P99 Cutoff):   {p99:.4f}")

plt.figure(figsize=(8, 4))
plt.hist(val_mse, bins=40, color='#10B981', alpha=0.7, label='Validation Normal MSE')
plt.axvline(p95, color='#F59E0B', linestyle='--', label=f'Medium Threshold (P95: {p95:.3f})')
plt.axvline(p99, color='#EF4444', linestyle='--', label=f'High Threshold (P99: {p99:.3f})')
plt.title('Validation Reconstruction Error Distribution & Thresholds')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Frequency')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Model Evaluation on Test Benchmark
Evaluate on held-out test data containing normal records and synthesized anomaly typologies.

In [ ]:
test_reconstructions = autoencoder.predict(X_test)
test_mse = np.mean(np.square(X_test - test_reconstructions), axis=1)
y_true = df_test['is_anomaly'].values
y_pred = (test_mse >= p95).astype(int)

roc_auc = roc_auc_score(y_true, test_mse)
pr_auc = average_precision_score(y_true, test_mse)

print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test PR-AUC:  {pr_auc:.4f}")
print("\nClassification Report (Flagged = Error >= P95):")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly']))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## 7. Feature-wise Error Decomposition
We inspect how reconstruction residuals localize onto specific anomalous features (e.g. rent vs income).

In [ ]:
mapping = preprocessor.get_feature_index_mapping()
sample_anom = df_anomaly.iloc[0:1]
sample_scaled = preprocessor.transform(sample_anom)
sample_recon = autoencoder.predict(sample_scaled)
col_errors = np.square(sample_scaled[0] - sample_recon[0])

feat_breakdown = {}
for feat, idxs in mapping.items():
    feat_breakdown[feat] = float(np.sum(col_errors[idxs]))

total_err = sum(feat_breakdown.values())
pct_breakdown = {k: round(v / total_err * 100, 2) for k, v in feat_breakdown.items()}

print("Sample Injected Anomaly Profile:")
print(sample_anom[['monthly_income', 'monthly_rent', 'credit_score', 'anomaly_type']].to_dict(orient='records')[0])
print("\nFeature Contribution to Reconstruction Error (%):")
for k, v in sorted(pct_breakdown.items(), key=lambda x: x[1], reverse=True):
    print(f" - {k}: {v}%")